<a href="https://colab.research.google.com/github/RyuichiSaito1/inflation-reddit-usa/blob/main/notebooks/roberta_large_fine_tuning_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Using L4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import auth
auth.authenticate_user()

# Install modern libraries matching DeBERTa environment
!pip install accelerate transformers datasets scikit-learn matplotlib

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    TrainerCallback
)
from datasets import Dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
import transformers

# Verify versions
print(f"Transformers version: {transformers.__version__}")

class TimeTrackerCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, model=None, **kwargs):
        self.start_time = time.time()

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        elapsed_time = time.time() - self.start_time
        print(f"Epoch {state.epoch} training time: {elapsed_time:.2f} seconds")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load dataset
df = pd.read_csv('/content/drive/MyDrive/world-inflation/data/reddit/production/2-main-prod-1039.csv', sep=',')
print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['inflation'].value_counts()}")

# Split with stratification to match DeBERTa/Llama experiments
train_df, val_df = train_test_split(
    df,
    test_size=0.25,
    random_state=42,
    stratify=df['inflation'] # Crucial for fair comparison
)

# Convert to Hugging Face Datasets
raw_train_dataset = Dataset.from_pandas(train_df)
raw_val_dataset = Dataset.from_pandas(val_df)

In [ ]:
model_name = "roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # RoBERTa uses max_position_embeddings=514 usually, but 512 is safe standard
    return tokenizer(
        examples['body'],
        padding="max_length",
        truncation=True,
        max_length=512
    )

# Apply tokenization
tokenized_train = raw_train_dataset.map(tokenize_function, batched=True)
tokenized_val = raw_val_dataset.map(tokenize_function, batched=True)

# Rename label column
tokenized_train = tokenized_train.rename_column("inflation", "labels")
tokenized_val = tokenized_val.rename_column("inflation", "labels")

# Remove unused columns
columns_to_keep = ['input_ids', 'attention_mask', 'labels']
tokenized_train = tokenized_train.remove_columns([c for c in tokenized_train.column_names if c not in columns_to_keep])
tokenized_val = tokenized_val.remove_columns([c for c in tokenized_val.column_names if c not in columns_to_keep])

# Set format
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    problem_type="single_label_classification"
)
model = model.to(device)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    accuracy = accuracy_score(labels, preds)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/world-inflation/data/model/roberta-large-fine-tuning-1039-rerun",
    logging_dir="/content/drive/MyDrive/world-inflation/data/model/roberta-large-fine-tuning-1039-rerun/logs",

    # Evaluation and saving strategy
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,

    # Hyperparameters aligned EXACTLY with your DeBERTa experiment
    learning_rate=2e-5,
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,

    # gradient_accumulation_steps is removed to match DeBERTa (Default is 1)
    # Effective Batch Size = 8 * 1 = 8

    # Regularization
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,

    # Use F1 score to determine the best model
    metric_for_best_model="eval_f1",
    greater_is_better=True,

    seed=42,

    # Precision settings (bf16 is recommended for T4/A100 GPUs)
    fp16=False,
    bf16=True,

    run_name="roberta-inflation-1040",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[TimeTrackerCallback()]
)

print("Starting training...")
trainer.train()

In [ ]:
# Visualize training history
logs = trainer.state.log_history

train_losses = []
eval_losses = []
epochs_train = []
epochs_eval = []

for log in logs:
    if 'loss' in log and 'epoch' in log:
        train_losses.append(log['loss'])
        epochs_train.append(log['epoch'])
    if 'eval_loss' in log and 'epoch' in log:
        eval_losses.append(log['eval_loss'])
        epochs_eval.append(log['epoch'])

plt.figure(figsize=(10, 6))
plt.plot(epochs_train, train_losses, label='Train Loss', marker='o')
plt.plot(epochs_eval, eval_losses, label='Validation Loss', marker='x')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Learning Curve: RoBERTa Large')
plt.legend()
plt.grid(True)
plt.show()